# step_renderer

> Main verify step renderer combining all dashboard components

In [ ]:
#| default_exp components.step_renderer

In [ ]:
#| export
from typing import Any, Optional

from fasthtml.common import Div, H2, Span

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles, badge_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import h, max_w, container
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons via factory
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_transcript_verify.html_ids import VerifyHtmlIds
from cjm_transcript_verify.models import VerificationResult, VerifyUrls
from cjm_transcript_verify.components.verification_summary import render_verification_summary
from cjm_transcript_verify.components.integrity_checks import render_integrity_checks
from cjm_transcript_verify.components.sample_segments import render_sample_segments

# Debug flag
DEBUG_VERIFY_RENDER = False

## Header Component

In [ ]:
#| export
def render_verify_header(
    all_passed:bool,  # Whether all integrity checks passed
) -> Any:  # Header element with title and status badge
    """Render the verify step header with status badge."""
    if all_passed:
        status_badge = Span(
            lucide_icon("circle-check", size=4),
            "All Checks Passed",
            cls=combine_classes(
                badge, badge_styles.outline, badge_sizes.md,
                "badge-success", flex_display, items.center, gap(1)
            )
        )
    else:
        status_badge = Span(
            lucide_icon("triangle-alert", size=4),
            "Issues Detected",
            cls=combine_classes(
                badge, badge_styles.outline, badge_sizes.md,
                "badge-warning", flex_display, items.center, gap(1)
            )
        )
    
    return Div(
        H2(
            "Verification Summary",
            cls=combine_classes(font_size._2xl, font_weight.bold)
        ),
        status_badge,
        cls=combine_classes(flex_display, justify.between, items.center, m.b(4))
    )

## Error State

In [ ]:
#| export
def render_verify_error(
    message:str="Unable to load verification data",  # Error message
) -> Any:  # Error display element
    """Render an error state for the verify step."""
    return Div(
        Div(
            lucide_icon("circle-alert", size=12, cls=str(text_dui.error)),
            H2(
                "Verification Error",
                cls=combine_classes(font_size.xl, font_weight.semibold, m.t(4))
            ),
            Span(
                message,
                cls=combine_classes(text_dui.base_content.opacity(70), m.t(2))
            ),
            cls=combine_classes(
                flex_display, flex_direction.col, items.center, justify.center,
                p(8), "text-center"
            )
        ),
        id=VerifyHtmlIds.VERIFY_CONTENT,
        cls=combine_classes(grow(), flex_display, items.center, justify.center)
    )

## Loading State

In [ ]:
#| export
def render_verify_loading() -> Any:  # Loading indicator
    """Render a loading state for the verify step."""
    return Div(
        Div(
            Span(cls="loading loading-spinner loading-lg"),
            Span(
                "Loading verification data...",
                cls=combine_classes(font_size.lg, m.t(4))
            ),
            cls=combine_classes(
                flex_display, flex_direction.col, items.center, justify.center
            )
        ),
        id=VerifyHtmlIds.VERIFY_CONTENT,
        cls=combine_classes(grow(), flex_display, items.center, justify.center)
    )

## Full Step Renderer

In [ ]:
#| export
def render_verify_step(
    result:Optional[VerificationResult]=None,  # Verification result or None for error
    urls:VerifyUrls=None,  # URL bundle for routes
    error:str="",  # Error message if result is None
) -> Any:  # Complete verify step component
    """Render the complete verify step with all dashboard components."""
    urls = urls or VerifyUrls()
    
    if DEBUG_VERIFY_RENDER:
        print(f"[VERIFY_RENDER] render_verify_step called")
        print(f"[VERIFY_RENDER] result: {result is not None}")
        print(f"[VERIFY_RENDER] error: {error}")
    
    # Handle error state
    if result is None:
        return Div(
            render_verify_error(error or "Unable to load verification data"),
            id=VerifyHtmlIds.VERIFY_CONTAINER,
            cls=combine_classes(
                container, max_w._5xl, m.x.auto,
                h.full, flex_display, flex_direction.col,
                p(4), gap(4)
            )
        )
    
    # Main content
    return Div(
        # Header with status badge
        render_verify_header(result.all_checks_passed),
        
        # Summary cards (document, segments, sources)
        render_verification_summary(result),
        
        # Integrity checks
        render_integrity_checks(result),
        
        # Sample segments with jump-to-index
        render_sample_segments(result, urls),
        
        id=VerifyHtmlIds.VERIFY_CONTAINER,
        cls=combine_classes(
            container, max_w._5xl, m.x.auto,
            flex_display, flex_direction.col,
            p(4), gap(4), overflow.y.auto
        )
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()